# Topic 6 — Train / Test / Validation
### Theory → tiny example → experiment.

**Why split data at all?** If you evaluate a model on the same data it trained on, it can just
memorize the answers — the score tells you nothing about real-world performance. You need data
the model has never seen to get an honest estimate.

**Why is this wrong?**
```text
Train → Test → Tune model  ❌
```
If you look at test performance and then go back and tweak hyperparameters, you're indirectly
"fitting" the test set — it stops being a fair, unseen evaluation.

**Why is this better?**
```text
Train → Validation → Final Test  ✅
```
- **Train set**: what the model learns from.
- **Validation set**: used repeatedly to tune hyperparameters / compare models.
- **Test set**: touched ONLY ONCE, at the very end, for the final honest score.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold, cross_val_score
)
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(0)

## 1. Basic random split

`train_test_split` shuffles and splits your data. `random_state` makes the split reproducible.

In [ ]:
X = np.arange(20).reshape(-1, 1)     # 20 "samples"
y = np.array([0]*15 + [1]*5)          # imbalanced-ish labels (15 zeros, 5 ones)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print("train size:", len(X_train), " test size:", len(X_test))
print("y_train:", y_train)
print("y_test:", y_test)
# Notice: with a plain random split, the class balance in train/test can drift from the original.

## 2. Stratified split

**Stratified** = preserve the same class proportions in each split as in the full dataset.
Critical for imbalanced data (like cyberbullying detection, where positives are often a minority).

In [ ]:
print("original class balance:", np.bincount(y) / len(y))

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y   # <-- the key argument
)

print("train class balance:", np.bincount(y_train_s) / len(y_train_s))
print("test class balance:", np.bincount(y_test_s) / len(y_test_s))
# Both splits now closely match the original 75/25 ratio.

## 3. Three-way split (train / validation / test)

`train_test_split` only splits into two, so call it twice to get three sets.

In [ ]:
X_full = np.arange(100).reshape(-1, 1)
y_full = rng.choice([0, 1], size=100, p=[0.8, 0.2])

# Step 1: split off the test set (never touch this until the very end)
X_temp, X_test, y_temp, y_test = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
)

# Step 2: split the remainder into train + validation
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp   # 0.25 of 80% = 20% of total
)

print("train:", len(X_train), " val:", len(X_val), " test:", len(X_test))
print("proportions:", len(X_train)/100, len(X_val)/100, len(X_test)/100)
# -> roughly 60% train, 20% validation, 20% test

## 4. Cross-validation & K-Fold

A single train/val split can be lucky or unlucky — the score depends on *which* rows landed where.
**K-Fold cross-validation**: split data into K equal parts ("folds"). Train on K-1 folds, validate
on the remaining fold. Repeat K times, so every row gets used for validation exactly once. Average
the K scores for a much more reliable estimate.

In [ ]:
X_cv = rng.normal(0, 1, size=(40, 3))
y_cv = rng.choice([0, 1], size=40)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(kf.split(X_cv)):
    print(f"fold {fold}: train size={len(train_idx)}, val size={len(val_idx)}")
    if fold == 0:
        print("  first fold's validation row indices:", val_idx)
# Every row appears in the validation set exactly once across all 5 folds.

## 5. Stratified K-Fold

Same idea as K-Fold, but preserves class balance within every fold — the version you should
default to for classification tasks with imbalanced classes.

In [ ]:
y_imbalanced = np.array([0]*35 + [1]*5)   # 40 samples, only 5 positives
X_imbalanced = rng.normal(0, 1, size=(40, 3))

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_imbalanced, y_imbalanced)):
    val_labels = y_imbalanced[val_idx]
    print(f"fold {fold}: val class counts -> 0s={sum(val_labels==0)}, 1s={sum(val_labels==1)}")
# Each fold gets roughly 1 positive sample -- a plain KFold could easily put ALL positives in one fold.

## 6. Putting it together: `cross_val_score`

Sklearn can run the whole "split, train, evaluate, repeat, average" loop in one line.

In [ ]:
model = LogisticRegression()

scores = cross_val_score(
    model, X_imbalanced, y_imbalanced,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="accuracy"
)

print("scores per fold:", scores)
print("mean accuracy:", scores.mean(), " +/- std:", scores.std())
# Reporting mean +/- std across folds is far more trustworthy than a single train/test split score.

## Exercise

In [ ]:
toy_df = pd.DataFrame({
    "feature": rng.normal(0, 1, 60),
    "label": rng.choice([0, 1], size=60, p=[0.85, 0.15]),
})

# --- Try it yourself ---
# 1. Do a plain (non-stratified) 80/20 split of toy_df and print the class balance of each side.
# 2. Now do a STRATIFIED 80/20 split and compare the class balances.
# 3. Run 5-fold StratifiedKFold on toy_df and print each fold's positive-class count.
# 4. Explain in one sentence why, for THIS dataset, StratifiedKFold matters more than plain KFold.

---
### Next up: **Topic 7 — Linear Regression** (your first real ML algorithm, from scratch + sklearn).

Say "next" when you're ready.